# PathSnap — Interactive GPS Maps with Folium

This notebook creates interactive maps for selected PathSnap trips. Each map can show:
- raw GPS observations
- road-network arcs
- the reference route reconstructed from `.route`
- a predicted route, when a prediction file is available
- GPS observations following unusually large timestamp gaps
- an optional GPS density heatmap

**Data contract:** `.track` columns are longitude, latitude, timestamp; `.nodes` columns are longitude, latitude; `.arcs` rows map arc IDs to `(from_node, to_node)`; `.route` is an ordered sequence of arc IDs. Node IDs and arc IDs are their respective zero-based row indices.

Only use `train/` and `student_test/`. Do not access the evaluator-test partition. Generated HTML maps are saved under `results/folium/`.


## 1. Imports and configuration

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import folium
from folium import FeatureGroup, LayerControl
from folium.plugins import HeatMap, Fullscreen

pd.set_option("display.max_columns", 30)

# The notebook normally lives in data_processing/. Support launching Jupyter
# either from that folder or from the repository root.
CWD = Path.cwd().resolve()
CANDIDATE_ROOTS = []
for base in (CWD, *CWD.parents):
    CANDIDATE_ROOTS.extend([base, base / "data_processing"])


def find_dataset_root():
    seen = set()
    for root in CANDIDATE_ROOTS:
        root = root.resolve()
        if root in seen:
            continue
        seen.add(root)

        datasets_dir = root / "datasets"
        if (datasets_dir / "train").is_dir() and (datasets_dir / "student_test").is_dir():
            return datasets_dir

        # Also support the extracted student archive layout.
        student_dir = root / "Student folder"
        if (student_dir / "train").is_dir() and (student_dir / "student_test").is_dir():
            return student_dir

    raise FileNotFoundError(
        "Could not locate a dataset containing both train/ and student_test/. "
        "Expected data_processing/datasets/{train,student_test} or an extracted "
        "'Student folder' directory."
    )


DATASET_ROOT = find_dataset_root()
TRAIN_PATH = DATASET_ROOT / "train"
STUDENT_TEST_PATH = DATASET_ROOT / "student_test"

# Keep generated HTML maps separate from EDA CSV outputs.
# If datasets/ is inside data_processing/, step up one more level to the
# repository root so generated maps land beside the existing results/ folder.
if DATASET_ROOT.name == "datasets" and DATASET_ROOT.parent.name == "data_processing":
    REPO_ROOT = DATASET_ROOT.parent.parent
else:
    REPO_ROOT = DATASET_ROOT.parent
MAPS_OUTPUT = REPO_ROOT / "results" / "folium"
MAPS_OUTPUT.mkdir(parents=True, exist_ok=True)

# Optional predictions CSV. Expected columns: trip_id, predicted_arc_ids.
# predicted_arc_ids may contain JSON/list text, e.g. "[12, 15, 18]".
PREDICTIONS_PATH = REPO_ROOT / "results" / "predictions" / "predicted_routes.csv"

print("Working directory:", CWD)
print("Dataset root:", DATASET_ROOT.resolve())
print("Training directory exists:", TRAIN_PATH.is_dir())
print("Student-test directory exists:", STUDENT_TEST_PATH.is_dir())
print("HTML output directory:", MAPS_OUTPUT.resolve())
print("Optional predictions file exists:", PREDICTIONS_PATH.is_file())


Working directory: /Users/maayamohan/Desktop/Maaya/sem5/ml/ML-18_11-E-7/data_processing
Dataset root: /Users/maayamohan/Desktop/Maaya/sem5/ml/ML-18_11-E-7/datasets
Training directory exists: True
Student-test directory exists: True
HTML output directory: /Users/maayamohan/Desktop/Maaya/sem5/ml/ML-18_11-E-7/results/folium
Optional predictions file exists: False


## 2. List available trips

In [2]:
def list_trip_dirs(partition_path):
    return sorted(
        p for p in partition_path.iterdir()
        if p.is_dir() and (p / f"{p.name}.track").exists()
    )

train_trips = list_trip_dirs(TRAIN_PATH)
student_test_trips = list_trip_dirs(STUDENT_TEST_PATH)

print(f"Training trips: {len(train_trips)}")
print(f"Student-test trips: {len(student_test_trips)}")

trip_catalogue = pd.DataFrame(
    [{"partition": "train", "trip_id": p.name} for p in train_trips]
    + [{"partition": "student_test", "trip_id": p.name} for p in student_test_trips]
)
display(trip_catalogue.head(15))

Training trips: 80
Student-test trips: 10


,partition,trip_id
0,train,00000001
1,train,00000002
2,train,00000003
3,train,00000004
4,train,00000005
5,train,00000006
6,train,00000008
7,train,00000010
8,train,00000011
9,train,00000012


## 3. Data-loading helpers

In [3]:
def read_trip_files(trip_id, partition="train"):
    """Load one trip; node and arc IDs are zero-based row indices."""
    partition_dirs = {
        "train": TRAIN_PATH,
        "student_test": STUDENT_TEST_PATH,
    }
    if partition not in partition_dirs:
        raise ValueError("partition must be 'train' or 'student_test'")

    trip_id = str(trip_id)
    trip_dir = partition_dirs[partition] / trip_id
    if not trip_dir.is_dir():
        raise FileNotFoundError(f"Trip directory not found: {trip_dir}")

    track_path = trip_dir / f"{trip_id}.track"
    nodes_path = trip_dir / f"{trip_id}.nodes"
    arcs_path = trip_dir / f"{trip_id}.arcs"
    route_path = trip_dir / f"{trip_id}.route"
    for path in (track_path, nodes_path, arcs_path, route_path):
        if not path.is_file():
            raise FileNotFoundError(f"Required trip file not found: {path}")

    # Whitespace-delimited source files; the regex must be a single \s escape.
    track = pd.read_csv(
        track_path, sep=r"\s+", header=None,
        names=["longitude", "latitude", "timestamp"],
    )
    nodes = pd.read_csv(
        nodes_path, sep=r"\s+", header=None,
        names=["longitude", "latitude"],
    )
    arcs = pd.read_csv(
        arcs_path, sep=r"\s+", header=None,
        names=["from_node", "to_node"],
    )
    route = pd.read_csv(route_path, sep=r"\s+", header=None).iloc[:, 0].astype(int).tolist()

    if track.empty:
        raise ValueError(f"Trip {trip_id} has no GPS observations")
    if track[["longitude", "latitude", "timestamp"]].isna().any().any():
        raise ValueError(f"Trip {trip_id} contains missing GPS coordinates or timestamps")

    # In these files, IDs are implicit row positions rather than explicit columns.
    arcs.index.name = "arc_id"
    nodes.index.name = "node_id"

    track = track.sort_values("timestamp").reset_index(drop=True)
    track["time_gap"] = track["timestamp"].diff()
    return track, nodes, arcs, route


def arc_endpoints(arc_id, nodes, arcs):
    """Return an arc as [[latitude, longitude], ...], or None if invalid."""
    if arc_id not in arcs.index:
        return None
    from_node = int(arcs.loc[arc_id, "from_node"])
    to_node = int(arcs.loc[arc_id, "to_node"])
    if from_node not in nodes.index or to_node not in nodes.index:
        return None
    start, end = nodes.loc[from_node], nodes.loc[to_node]
    return [
        [float(start["latitude"]), float(start["longitude"])],
        [float(end["latitude"]), float(end["longitude"])],
    ]


def route_coordinates(route_arc_ids, nodes, arcs):
    """Convert ordered arc IDs into a route polyline and count invalid arc IDs."""
    coords = []
    missing = 0
    for arc_id in route_arc_ids:
        endpoints = arc_endpoints(int(arc_id), nodes, arcs)
        if endpoints is None:
            missing += 1
            continue
        if not coords:
            coords.extend(endpoints)
        elif np.allclose(coords[-1], endpoints[0], atol=1e-7):
            coords.append(endpoints[1])
        elif np.allclose(coords[-1], endpoints[1], atol=1e-7):
            # Handle a route that traverses this arc opposite to its stored direction.
            coords.append(endpoints[0])
        else:
            # Preserve a discontinuity rather than silently changing arc order.
            coords.extend(endpoints)
    return coords, missing


def load_predicted_arc_ids(trip_id, predictions_path=PREDICTIONS_PATH):
    """Load optional predicted arc IDs; return None if no prediction is available."""
    if not predictions_path.is_file():
        return None
    predictions = pd.read_csv(predictions_path, dtype={"trip_id": str})
    required = {"trip_id", "predicted_arc_ids"}
    if not required.issubset(predictions.columns):
        raise ValueError(f"Prediction CSV must contain columns: {sorted(required)}")

    rows = predictions.loc[predictions["trip_id"].astype(str) == str(trip_id)]
    if rows.empty:
        return None
    value = rows.iloc[0]["predicted_arc_ids"]
    if pd.isna(value) or not str(value).strip():
        return None
    if isinstance(value, str):
        try:
            value = json.loads(value)
        except json.JSONDecodeError:
            value = [part.strip() for part in value.strip("[]").split(",") if part.strip()]
    if not isinstance(value, (list, tuple)):
        raise ValueError("predicted_arc_ids must be a list or a string containing a list")
    return [int(arc_id) for arc_id in value]


## 4. Inspect one trip before mapping

In [4]:
# Start with a known training trip from the EDA work.
TRIP_ID = "00000001"
PARTITION = "train"

track, nodes, arcs, reference_route = read_trip_files(TRIP_ID, PARTITION)

print(f"Trip: {TRIP_ID} ({PARTITION})")
print(f"GPS observations: {len(track):,}")
print(f"Road nodes: {len(nodes):,}")
print(f"Road arcs: {len(arcs):,}")
print(f"Reference route arcs: {len(reference_route):,}")
display(track.head())
display(nodes.head())
display(arcs.head())
print("First reference arc IDs:", reference_route[:10])

Trip: 00000001 (train)
GPS observations: 1,315
Road nodes: 88,809
Road arcs: 165,790
Reference route arcs: 525


,longitude,latitude,timestamp,time_gap
0,8.595203,49.844053,1.0,NaN
1,8.595294,49.844441,2.0,1.0
2,8.595352,49.844883,3.0,1.0
3,8.595390,49.845379,4.0,1.0
4,8.595587,49.845750,5.0,1.0


,longitude,latitude
node_id,,
0,8.581679,49.857880
1,8.584637,49.858295
2,8.580173,49.858567
3,8.584392,49.859119
4,8.584935,49.857396


,from_node,to_node
arc_id,,
0,0,281
1,0,1
2,0,5
3,1,0
4,1,40750


First reference arc IDs: [2641, 64973, 133948, 133949, 139208, 133950, 8970, 133951, 64139, 139211]


## 5. Build an interactive map for one trip

In [5]:
def create_trip_map(
    trip_id,
    partition="train",
    show_gps_points=True,
    show_road_network=True,
    show_reference_route=True,
    show_predicted_route=True,
    show_anomalies=True,
    show_heatmap=False,
    max_road_arcs=5000,
    gps_point_radius=2,
):
    track, nodes, arcs, reference_route = read_trip_files(trip_id, partition)

    # Center map on the GPS observations.
    center = [track["latitude"].mean(), track["longitude"].mean()]
    fmap = folium.Map(
        location=center,
        zoom_start=13,
        tiles="OpenStreetMap",
        control_scale=True,
    )
    Fullscreen(position="topright").add_to(fmap)

    # Raw GPS observations.
    if show_gps_points:
        gps_layer = FeatureGroup(name="Raw GPS observations", show=True)
        # CircleMarker keeps the HTML reasonably lightweight for long trips.
        for i, row in track.iterrows():
            popup = (
                f"Observation: {i}<br>"
                f"Longitude: {row.longitude:.6f}<br>"
                f"Latitude: {row.latitude:.6f}<br>"
                f"Timestamp: {row.timestamp:g} s"
            )
            folium.CircleMarker(
                location=[row.latitude, row.longitude],
                radius=gps_point_radius,
                color="#2471A3",
                weight=1,
                fill=True,
                fill_opacity=0.65,
                popup=folium.Popup(popup, max_width=280),
            ).add_to(gps_layer)
        gps_layer.add_to(fmap)

    # Road-network geometry. Join arc endpoints in vectorized pandas operations
    # before drawing a capped local subset; this avoids hundreds of thousands of
    # repeated .loc lookups for large road networks.
    if show_road_network:
        road_layer = FeatureGroup(name="Road network", show=False)
        from_nodes = nodes.reindex(arcs["from_node"].astype(int).to_numpy())
        to_nodes = nodes.reindex(arcs["to_node"].astype(int).to_numpy())
        road_geometry = pd.DataFrame({
            "arc_id": arcs.index.to_numpy(),
            "from_lon": from_nodes["longitude"].to_numpy(),
            "from_lat": from_nodes["latitude"].to_numpy(),
            "to_lon": to_nodes["longitude"].to_numpy(),
            "to_lat": to_nodes["latitude"].to_numpy(),
        }).dropna()

        if len(road_geometry) > max_road_arcs:
            lon_min, lon_max = track.longitude.min(), track.longitude.max()
            lat_min, lat_max = track.latitude.min(), track.latitude.max()
            pad_lon = max((lon_max - lon_min) * 0.15, 0.002)
            pad_lat = max((lat_max - lat_min) * 0.15, 0.002)
            nearby = (
                (road_geometry[["from_lon", "to_lon"]].max(axis=1) >= lon_min - pad_lon)
                & (road_geometry[["from_lon", "to_lon"]].min(axis=1) <= lon_max + pad_lon)
                & (road_geometry[["from_lat", "to_lat"]].max(axis=1) >= lat_min - pad_lat)
                & (road_geometry[["from_lat", "to_lat"]].min(axis=1) <= lat_max + pad_lat)
            )
            road_geometry = road_geometry.loc[nearby].head(max_road_arcs)

        for arc in road_geometry.itertuples(index=False):
            folium.PolyLine(
                [[arc.from_lat, arc.from_lon], [arc.to_lat, arc.to_lon]],
                color="#7F8C8D", weight=1, opacity=0.45,
                tooltip=f"Road arc {arc.arc_id}",
            ).add_to(road_layer)
        road_layer.add_to(fmap)
        print(f"Road arcs drawn: {len(road_geometry):,} (network contains {len(arcs):,})")

    # Reference route from the ordered .route arc IDs.
    if show_reference_route:
        reference_coords, missing_reference = route_coordinates(reference_route, nodes, arcs)
        if len(reference_coords) >= 2:
            route_layer = FeatureGroup(name="Reference route", show=True)
            folium.PolyLine(
                reference_coords, color="#229954", weight=5, opacity=0.9,
                tooltip="Reference route"
            ).add_to(route_layer)
            folium.Marker(
                reference_coords[0], tooltip="Reference route — start",
                icon=folium.Icon(color="green", icon="play")
            ).add_to(route_layer)
            folium.Marker(
                reference_coords[-1], tooltip="Reference route — end",
                icon=folium.Icon(color="darkgreen", icon="stop")
            ).add_to(route_layer)
            route_layer.add_to(fmap)
        print(f"Reference route arcs missing from network: {missing_reference}")

    # Predicted route is optional until the API/model output is available.
    if show_predicted_route:
        predicted_arc_ids = load_predicted_arc_ids(trip_id)
        if predicted_arc_ids is not None:
            predicted_coords, missing_predicted = route_coordinates(predicted_arc_ids, nodes, arcs)
            if len(predicted_coords) >= 2:
                predicted_layer = FeatureGroup(name="Predicted route", show=True)
                folium.PolyLine(
                    predicted_coords, color="#8E44AD", weight=5, opacity=0.85,
                    tooltip="Predicted route"
                ).add_to(predicted_layer)
                predicted_layer.add_to(fmap)
            print(f"Predicted route arcs missing from network: {missing_predicted}")
        else:
            print("No prediction found. Predicted route layer skipped.")

    # Mark the GPS point immediately after each large timestamp gap.
    if show_anomalies:
        anomaly_rows = track.loc[track["time_gap"] > 5]
        if not anomaly_rows.empty:
            anomaly_layer = FeatureGroup(name="GPS gaps over 5 seconds", show=True)
            for idx, row in anomaly_rows.iterrows():
                folium.CircleMarker(
                    [row.latitude, row.longitude],
                    radius=7, color="#E74C3C", weight=2,
                    fill=True, fill_color="#E74C3C", fill_opacity=0.9,
                    popup=(
                        f"Observation: {idx}<br>"
                        f"Timestamp gap: {row.time_gap:g} seconds"
                    ),
                    tooltip=f"Gap: {row.time_gap:g} s",
                ).add_to(anomaly_layer)
            anomaly_layer.add_to(fmap)

    if show_heatmap:
        heat_layer = FeatureGroup(name="GPS density heatmap", show=False)
        heat_data = track[["latitude", "longitude"]].dropna().values.tolist()
        HeatMap(heat_data, radius=10, blur=12, min_opacity=0.25).add_to(heat_layer)
        heat_layer.add_to(fmap)

    LayerControl(collapsed=False).add_to(fmap)
    fmap.fit_bounds([
        [track.latitude.min(), track.longitude.min()],
        [track.latitude.max(), track.longitude.max()],
    ])
    return fmap


trip_map = create_trip_map(
    TRIP_ID,
    PARTITION,
    show_heatmap=False,
)
display(trip_map)

Road arcs drawn: 5,000 (network contains 165,790)
Reference route arcs missing from network: 0
No prediction found. Predicted route layer skipped.


## 6. Save the interactive HTML map

In [6]:
output_file = MAPS_OUTPUT / f"{PARTITION}_{TRIP_ID}.html"
trip_map.save(str(output_file))
print("Saved map:", output_file.resolve())

Saved map: /Users/maayamohan/Desktop/Maaya/sem5/ml/ML-18_11-E-7/results/folium/train_00000001.html


## 7. Compare additional trips

In [7]:
# Choose a small set first. Add more IDs once the map style and performance look good.
TRIPS_TO_MAP = [
    ("train", "00000001"),
    ("train", "00000090"),  # trip with the largest timestamp gap found during EDA
    ("train", "00000078"),  # another trip with a very large timestamp gap
    ("student_test", "00000000"),
]

saved_maps = []
for partition, trip_id in TRIPS_TO_MAP:
    trip_dir = (TRAIN_PATH if partition == "train" else STUDENT_TEST_PATH) / trip_id
    if not trip_dir.exists():
        print(f"Skipping missing trip: {partition}/{trip_id}")
        continue

    fmap = create_trip_map(
        trip_id,
        partition=partition,
        show_road_network=True,
        show_reference_route=True,
        show_predicted_route=True,
        show_anomalies=True,
        show_heatmap=False,
    )
    path = MAPS_OUTPUT / f"{partition}_{trip_id}.html"
    fmap.save(str(path))
    saved_maps.append(path)
    print("Saved:", path.name)

print(f"\nMaps saved: {len(saved_maps)}")

Road arcs drawn: 5,000 (network contains 165,790)
Reference route arcs missing from network: 0
No prediction found. Predicted route layer skipped.
Saved: train_00000001.html
Road arcs drawn: 4,693 (network contains 7,942)
Reference route arcs missing from network: 0
No prediction found. Predicted route layer skipped.
Saved: train_00000090.html
Road arcs drawn: 5,000 (network contains 151,458)
Reference route arcs missing from network: 0
No prediction found. Predicted route layer skipped.
Saved: train_00000078.html
Road arcs drawn: 5,000 (network contains 18,196)
Reference route arcs missing from network: 0
No prediction found. Predicted route layer skipped.
Saved: student_test_00000000.html

Maps saved: 4


## 8. Optional: create a GPS density heatmap

The heatmap is useful for seeing where observations cluster across a partition. It is not a route: it shows point density. To keep the map responsive, sample points if the partition is large.

In [8]:
def create_partition_heatmap(partition="train", sample_per_trip=1000):
    """Create a sampled GPS density map for one allowed data partition."""
    partition_paths = {
        "train": TRAIN_PATH,
        "student_test": STUDENT_TEST_PATH,
    }
    if partition not in partition_paths:
        raise ValueError("partition must be 'train' or 'student_test'")
    if sample_per_trip < 1:
        raise ValueError("sample_per_trip must be at least 1")

    partition_path = partition_paths[partition]
    sampled_points = []

    for trip_dir in list_trip_dirs(partition_path):
        track_file = trip_dir / f"{trip_dir.name}.track"
        data = pd.read_csv(
            track_file, sep=r"\s+", header=None,
            names=["longitude", "latitude", "timestamp"],
        )
        if len(data) > sample_per_trip:
            sample_indices = np.linspace(0, len(data) - 1, sample_per_trip, dtype=int)
            data = data.iloc[sample_indices]
        sampled_points.extend(data[["latitude", "longitude"]].dropna().values.tolist())

    if not sampled_points:
        raise ValueError(f"No GPS points found for partition: {partition}")

    center = [
        float(np.mean([point[0] for point in sampled_points])),
        float(np.mean([point[1] for point in sampled_points])),
    ]
    fmap = folium.Map(location=center, zoom_start=4, tiles="CartoDB positron")
    HeatMap(sampled_points, radius=7, blur=10, min_opacity=0.2).add_to(fmap)
    return fmap, len(sampled_points)


# Run only if a partition-wide density view is useful.
# heatmap, sampled_count = create_partition_heatmap("train")
# display(heatmap)
# heatmap_path = MAPS_OUTPUT / "train_gps_heatmap.html"
# heatmap.save(str(heatmap_path))
# print(f"Saved {sampled_count:,} sampled points to {heatmap_path}")


## 9. Notes and interpretation

- The GPS points are raw observations from `.track`.
- The road network is drawn from `.nodes` and `.arcs`; the arc ID is the row index in `.arcs`.
- The reference route is the ordered arc-ID sequence in `.route`, joined using each arc's directed endpoints.
- The predicted route is only drawn when `results/predictions/predicted_routes.csv` exists and contains `trip_id` and `predicted_arc_ids`.
- Red markers identify observations whose timestamp gap from the previous observation exceeds 5 seconds. This is a visualization threshold consistent with the EDA investigation; it does not itself prove the GPS point is erroneous.
- The HTML files are interactive and can be opened in a browser. Commit the generation notebook and selected HTML deliverables, subject to repository size.